# Customer Segmentation — UCI Online Retail Dataset

**Technique:** RFM Feature Engineering + K-Means Clustering  
**Dataset:** UCI Online Retail (541,909 transactions, Dec 2010 – Dec 2011)  
**Author:** Anthony  
**Environment:** Python 3.11 · Anaconda · conda env `proxy_coverage`

---

## Objective

Segment customers into meaningful behavioural groups using three core purchasing signals:

- **Recency** — how recently did they buy?
- **Frequency** — how often do they buy?
- **Monetary** — how much do they spend?

The output is four actionable segments that can drive targeted marketing, retention, and re-engagement strategies.

---

## Workflow

1. Load & inspect data
2. Clean & validate
3. Engineer RFM features
4. Handle skew + scale
5. Select optimal K (Elbow + Silhouette)
6. Fit final K-Means model
7. Assign segment labels programmatically
8. Visualise clusters
9. Save outputs

## Step 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Reproducibility
RANDOM_STATE = 42

# Paths — relative so the notebook runs on any machine
DATA_PATH    = Path('../data/online_retail.xlsx')
OUTPUT_PATH  = Path('../outputs')
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print('All libraries loaded.')
print(f'Data:    {DATA_PATH}')
print(f'Outputs: {OUTPUT_PATH}')

## Step 2 — Load & Inspect

In [ ]:
df = pd.read_excel(DATA_PATH)

print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print()
print('Data types:')
print(df.dtypes)
print()
print('Date range:')
print(df['InvoiceDate'].min(), 'to', df['InvoiceDate'].max())

## Step 3 — Clean & Validate

Removals:
- Rows with no `CustomerID` (can't build RFM without a customer key)
- Cancelled orders — invoices prefixed with `'C'`
- Rows where `Quantity` or `UnitPrice` ≤ 0 (returns, data errors)

In [ ]:
raw_shape = df.shape

df = df.dropna(subset=['CustomerID'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

df['CustomerID'] = df['CustomerID'].astype(int)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

print(f'Raw shape:     {raw_shape}')
print(f'Cleaned shape: {df.shape}')
print(f'Rows removed:  {raw_shape[0] - df.shape[0]:,}')
print()
print('Missing values after cleaning:')
print(df.isnull().sum())
df.head()

## Step 4 — Engineer RFM Features

Reference date is set to **one day after the last transaction** — a standard convention that ensures the most recent customer gets a recency of 1 (not 0).

In [ ]:
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Reference date: {reference_date}')
print(f'Date range in data: {df["InvoiceDate"].min()} to {df["InvoiceDate"].max()}')

In [ ]:
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

print(f'RFM shape: {rfm.shape}  (one row per customer)')
rfm.describe().round(2)

**Distribution check** — RFM features are typically right-skewed. We'll visualise before transforming.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['Recency', 'Frequency', 'Monetary']):
    ax.hist(rfm[col], bins=50, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(f'{col} distribution', fontsize=12)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('RFM — Raw Distributions (note right skew)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Step 5 — Handle Skew + Scale

`log1p` (i.e. log(1 + x)) compresses the long tail without breaking on zero values. `StandardScaler` then centres each feature to mean=0, std=1 so K-Means treats all three dimensions equally.

In [ ]:
rfm_log = rfm.copy()
rfm_log['Recency']   = np.log1p(rfm['Recency'])
rfm_log['Frequency'] = np.log1p(rfm['Frequency'])
rfm_log['Monetary']  = np.log1p(rfm['Monetary'])

scaler     = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log[['Recency', 'Frequency', 'Monetary']])
rfm_scaled = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])

print('Scaled data — confirm mean ≈ 0, std ≈ 1:')
rfm_scaled.describe().round(3)

## Step 6 — Select Optimal K

Two complementary methods:
- **Elbow method** — look for the point where adding more clusters yields diminishing returns in inertia
- **Silhouette score** — measures how well-separated clusters are (higher = better, max 1.0)

In [ ]:
k_range    = range(2, 11)
inertia    = []
sil_scores = []

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels))

print('K  |  Inertia   |  Silhouette')
print('-' * 35)
for k, ine, sil in zip(k_range, inertia, sil_scores):
    print(f'{k:2d} |  {ine:8.2f}  |  {sil:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertia, marker='o', color='steelblue', linewidth=2)
axes[0].axvline(x=4, color='tomato', linestyle='--', alpha=0.7, label='Selected k=4')
axes[0].set_xlabel('Number of clusters (K)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow method', fontsize=13)
axes[0].set_xticks(list(k_range))
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_range, sil_scores, marker='o', color='tomato', linewidth=2)
axes[1].axvline(x=4, color='steelblue', linestyle='--', alpha=0.7, label='Selected k=4')
axes[1].set_xlabel('Number of clusters (K)')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette score', fontsize=13)
axes[1].set_xticks(list(k_range))
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Selecting optimal K', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**k=4 selected** — the elbow flattens noticeably after 4, and k=4 balances statistical separation with business interpretability (four segments map cleanly to actionable strategies).

## Step 7 — Fit Final K-Means Model

In [ ]:
km_final = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
rfm['Cluster'] = km_final.fit_predict(rfm_scaled)

print('Customers per cluster:')
print(rfm['Cluster'].value_counts().sort_index())

## Step 8 — Assign Segment Labels Programmatically

K-Means cluster numbers (0, 1, 2, 3) are **arbitrary** — they are not stable across runs or data changes. Labels must be assigned based on the actual RFM profile of each cluster, not hardcoded to cluster numbers.

Logic:
- **VIP** → highest Monetary value
- **Promising** → lowest Recency among the remaining clusters (recently active)
- **Hibernating** → highest Recency among the remaining clusters (longest since last purchase)
- **Needs Attention** → the remaining cluster

In [ ]:
cluster_profile = rfm.groupby('Cluster').agg(
    Recency   = ('Recency',   'mean'),
    Frequency = ('Frequency', 'mean'),
    Monetary  = ('Monetary',  'mean'),
    Count     = ('CustomerID','count')
).round(1)

print('Raw cluster profiles (before labelling):')
print(cluster_profile)

In [ ]:
# Assign labels based on RFM profile — never hardcode cluster numbers
vip_cluster         = cluster_profile['Monetary'].idxmax()
remaining           = cluster_profile.index[cluster_profile.index != vip_cluster].tolist()
promising_cluster   = cluster_profile.loc[remaining, 'Recency'].idxmin()   # most recent
remaining.remove(promising_cluster)
hibernating_cluster = cluster_profile.loc[remaining, 'Recency'].idxmax()   # longest silent
remaining.remove(hibernating_cluster)
needs_attention_cluster = remaining[0]                                       # the last one

segment_map = {
    vip_cluster:             'VIP',
    promising_cluster:       'Promising',
    hibernating_cluster:     'Hibernating',
    needs_attention_cluster: 'Needs Attention',
}

rfm['Segment'] = rfm['Cluster'].map(segment_map)

print('Cluster → Segment mapping:')
for cluster, label in sorted(segment_map.items()):
    print(f'  Cluster {cluster} → {label}')

print()
print('Customers per segment:')
print(rfm['Segment'].value_counts())

In [ ]:
# Labelled cluster profiles — the core deliverable
labelled_profile = rfm.groupby('Segment').agg(
    Recency   = ('Recency',   'mean'),
    Frequency = ('Frequency', 'mean'),
    Monetary  = ('Monetary',  'mean'),
    Count     = ('CustomerID','count')
).round(1).sort_values('Monetary', ascending=False)

print('Labelled segment profiles:')
print(labelled_profile)

## Step 9 — Visualise Clusters

In [ ]:
# Consistent colour palette keyed on segment name — not cluster number
PALETTE = {
    'VIP':             '#185FA5',
    'Promising':       '#0F6E56',
    'Needs Attention': '#BA7517',
    'Hibernating':     '#888780',
}

segment_order = ['VIP', 'Promising', 'Needs Attention', 'Hibernating']
plot_data     = labelled_profile.reindex(segment_order)
colors        = [PALETTE[s] for s in segment_order]

In [ ]:
# Bar chart — RFM profiles per segment
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

metrics = [
    ('Recency',   'Avg recency (days)',    'Lower = better'),
    ('Frequency', 'Avg order frequency',   'Higher = better'),
    ('Monetary',  'Avg monetary spend (£)', 'Higher = better'),
]

for ax, (col, ylabel, note) in zip(axes, metrics):
    bars = ax.bar(segment_order, plot_data[col], color=colors, width=0.55)
    ax.set_title(f'{col}\n({note})', fontsize=12)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.tick_params(axis='x', rotation=15)
    ax.grid(axis='y', alpha=0.3)
    # Value labels on bars
    for bar, val in zip(bars, plot_data[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + bar.get_height()*0.02,
                f'{val:,.0f}', ha='center', va='bottom', fontsize=9, color='#333')

plt.suptitle('Customer segments — average RFM profile', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot — Recency vs Monetary coloured by segment
fig, ax = plt.subplots(figsize=(10, 6))

for segment in segment_order:
    subset = rfm[rfm['Segment'] == segment]
    ax.scatter(
        subset['Recency'], subset['Monetary'],
        c=PALETTE[segment], label=segment,
        alpha=0.5, s=25, edgecolors='none'
    )

ax.set_xlabel('Recency (days since last purchase)', fontsize=11)
ax.set_ylabel('Monetary — total spend (£)', fontsize=11)
ax.set_title('Customer segments — Recency vs Monetary', fontsize=13, fontweight='bold')
ax.legend(title='Segment', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue concentration — customers vs revenue share
revenue_by_segment = rfm.groupby('Segment')['Monetary'].sum()
count_by_segment   = rfm.groupby('Segment')['CustomerID'].count()
total_revenue      = revenue_by_segment.sum()
total_customers    = count_by_segment.sum()

rev_pct   = (revenue_by_segment / total_revenue * 100).reindex(segment_order)
cust_pct  = (count_by_segment   / total_customers * 100).reindex(segment_order)

x      = np.arange(len(segment_order))
width  = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, rev_pct,  width, label='Revenue %',  color=[PALETTE[s] for s in segment_order])
bars2 = ax.bar(x + width/2, cust_pct, width, label='Customer %', color='#cccccc')

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.5, f'{h:.1f}%',
            ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(segment_order)
ax.set_ylabel('Share (%)', fontsize=11)
ax.set_title('Revenue concentration — customer % vs revenue %', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nKey insight: VIP customers are {cust_pct["VIP"]:.1f}% of the base but drive {rev_pct["VIP"]:.1f}% of revenue.')

## Step 10 — Merge Segments onto Transactions & Save Outputs

In [ ]:
# Merge segment labels onto the full cleaned transaction dataframe
df_segmented = df.merge(
    rfm[['CustomerID', 'Cluster', 'Segment']],
    on='CustomerID',
    how='left'
)

print(f'Segmented transactions shape: {df_segmented.shape}')
print()
print('Transactions per segment:')
print(df_segmented['Segment'].value_counts())
df_segmented.head()

In [ ]:
# Save outputs
rfm_out = OUTPUT_PATH / 'rfm_segments.csv'
tx_out  = OUTPUT_PATH / 'transactions_segmented.csv'

rfm.to_csv(rfm_out, index=False)
df_segmented.to_csv(tx_out, index=False)

print('Files saved:')
print(f'  {rfm_out}  ({len(rfm):,} rows — one per customer)')
print(f'  {tx_out}  ({len(df_segmented):,} rows — all transactions)')

## Summary

| Segment | Customers | Revenue share | Avg recency | Avg frequency | Avg spend |
|---|---|---|---|---|---|
| VIP | 716 | 64.9% | 12 days | 13.7 orders | £8,074 |
| Promising | 837 | 5.2% | 18 days | 2.1 orders | £552 |
| Needs Attention | 1,173 | 23.7% | 71 days | 4.1 orders | £1,803 |
| Hibernating | 1,612 | 6.2% | 183 days | 1.3 orders | £344 |

### Recommended actions

| Segment | Strategy |
|---|---|
| **VIP** | Loyalty rewards, early access, dedicated account management. Protect at all costs. |
| **Promising** | Cross-sell campaigns, volume incentives, onboarding sequences to increase frequency. |
| **Needs Attention** | Win-back offers, personalised recommendations — they have spend history worth recovering. |
| **Hibernating** | Deep discount reactivation attempt; sunset unresponsive contacts to protect list health. |